## 📋 Estrutura do Relatório

1. **Trilha 1: Estrutura do Mercado** · cargos e senioridade
2. **Trilha 2: Remuneração e Valorização** · faixas salariais por senioridade
3. **Trilha 3: Perfil dos Profissionais** · gênero, região e modelo de trabalho
4. **Trilha 4: Tecnologias e Inteligência Artificial** · adoção de SQL, Python e IA
5. **Trilha 5: Oportunidades e Recomendações** · riscos, recomendações e conclusão

**Pergunta norteadora:** Como uma instituição financeira de grande porte pode contratar, capacitar e investir em Dados, Analytics e Inteligência Artificial a partir do que o mercado brasileiro de dados mostra hoje?

**Base de dados:** pesquisa State of Data Brasil (Data Hackers e Bain), três últimas edições (2023, 2024 e 2025/26), camada Gold com 8 tabelas.

### Bibliotecas necessárias para fazer o Tech Challenge

In [1]:
# bibliotecas importantes
import os
import sys
import subprocess
import decimal
import textwrap

# Instalando PySpark
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)

# Manipulação dos dados
import pandas as pd
import numpy as np

# Spark (processamento dos dados)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Visualização dos dados
import plotly.express as px
import plotly.graph_objects as go

# Mostrar tabelas formatadas no notebook
from IPython.display import display

In [2]:
# Cria a sessão do Spark
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("tech-challenge-fase3")
    .getOrCreate()
)

print("Spark versão:", spark.version)

Spark versão: 4.0.4


In [3]:
# Ler as bases de dados - camada Gold

df_cargo = spark.read.csv("perfil_cargo.csv", header=True, inferSchema=True)
df_genero = spark.read.csv("perfil_genero.csv", header=True, inferSchema=True)
df_modelo_trabalho = spark.read.csv("perfil_modelo_trabalho.csv", header=True, inferSchema=True)
df_regiao = spark.read.csv("perfil_regiao.csv", header=True, inferSchema=True)
df_remuneracao = spark.read.csv("perfil_remuneracao.csv", header=True, inferSchema=True)
df_senioridade = spark.read.csv("perfil_senioridade.csv", header=True, inferSchema=True)
df_tecnologias = spark.read.csv("perfil_tecnologias.csv", header=True, inferSchema=True)
df_uso_ia = spark.read.csv("perfil_uso_ia.csv", header=True, inferSchema=True)

In [4]:
# Confere o tamanho de cada tabela (linhas e colunas)
print("perfil_cargo:", df_cargo.count(), "linhas e", len(df_cargo.columns), "colunas")
print("perfil_genero:", df_genero.count(), "linhas e", len(df_genero.columns), "colunas")
print("perfil_modelo_trabalho:", df_modelo_trabalho.count(), "linhas e", len(df_modelo_trabalho.columns), "colunas")
print("perfil_regiao:", df_regiao.count(), "linhas e", len(df_regiao.columns), "colunas")
print("perfil_remuneracao:", df_remuneracao.count(), "linhas e", len(df_remuneracao.columns), "colunas")
print("perfil_senioridade:", df_senioridade.count(), "linhas e", len(df_senioridade.columns), "colunas")
print("perfil_tecnologias:", df_tecnologias.count(), "linhas e", len(df_tecnologias.columns), "colunas")
print("perfil_uso_ia:", df_uso_ia.count(), "linhas e", len(df_uso_ia.columns), "colunas")

perfil_cargo: 45 linhas e 5 colunas
perfil_genero: 12 linhas e 5 colunas
perfil_modelo_trabalho: 9 linhas e 5 colunas
perfil_regiao: 15 linhas e 5 colunas
perfil_remuneracao: 113 linhas e 6 colunas
perfil_senioridade: 10 linhas e 5 colunas
perfil_tecnologias: 18 linhas e 8 colunas
perfil_uso_ia: 12 linhas e 7 colunas


In [5]:
# visualização das tabelas (5 primeiras linhas de cada uma)

print(' CARGO ')
df_cargo.show(5, truncate=False)

print(' GENERO ')
df_genero.show(5, truncate=False)

print(' MODELO DE TRABALHO ')
df_modelo_trabalho.show(5, truncate=False)

print(' REGIAO ')
df_regiao.show(5, truncate=False)

print(' REMUNERACAO ')
df_remuneracao.show(5, truncate=False)

print(' SENIORIDADE ')
df_senioridade.show(5, truncate=False)

print(' TECNOLOGIAS ')
df_tecnologias.show(5, truncate=False)

print(' USO DE IA ')
df_uso_ia.show(5, truncate=False)

 CARGO 
+------------+-----------------------------------------------------------+----------+-----------------------+----------+
|ano_pesquisa|cargo                                                      |quantidade|total_respostas_validas|percentual|
+------------+-----------------------------------------------------------+----------+-----------------------+----------+
|2023        |Cientista de Dados/Data Scientist                          |687       |3857                   |17.81     |
|2023        |Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas|105       |3857                   |2.72      |
|2023        |Professor/Pesquisador                                      |28        |3857                   |0.73      |
|2023        |Analytics Engineer                                         |139       |3857                   |3.6       |
|2023        |DBA/Administrador de Banco de Dados                        |20        |3857                   |0.52      |
+------------+----------

In [6]:
# Verificando os tipos de dados

print(' CARGO ')
df_cargo.printSchema()

print(' GENERO ')
df_genero.printSchema()

print(' MODELO DE TRABALHO ')
df_modelo_trabalho.printSchema()

print(' REGIAO ')
df_regiao.printSchema()

print(' REMUNERACAO ')
df_remuneracao.printSchema()

print(' SENIORIDADE ')
df_senioridade.printSchema()

print(' TECNOLOGIAS ')
df_tecnologias.printSchema()

print(' USO DE IA ')
df_uso_ia.printSchema()

 CARGO 
root
 |-- ano_pesquisa: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- total_respostas_validas: integer (nullable = true)
 |-- percentual: double (nullable = true)

 GENERO 
root
 |-- ano_pesquisa: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- total_respostas_validas: integer (nullable = true)
 |-- percentual: double (nullable = true)

 MODELO DE TRABALHO 
root
 |-- ano_pesquisa: string (nullable = true)
 |-- modelo_trabalho: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- total_respostas_validas: integer (nullable = true)
 |-- percentual: double (nullable = true)

 REGIAO 
root
 |-- ano_pesquisa: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- total_respostas_validas: integer (nullable = true)
 |-- percentual: double (nullable = true)

 REMUNERACAO 
root
 |-- ano_p

In [8]:
# Verificando valores nulos em cada tabela
# dropna() remove as linhas que têm algum valor vazio.
# Então: total de linhas - linhas sem nulo = linhas com nulo

print("cargo:", df_cargo.count() - df_cargo.dropna().count(), "linhas com nulo")
print("genero:", df_genero.count() - df_genero.dropna().count(), "linhas com nulo")
print("modelo_trabalho:", df_modelo_trabalho.count() - df_modelo_trabalho.dropna().count(), "linhas com nulo")
print("regiao:", df_regiao.count() - df_regiao.dropna().count(), "linhas com nulo")
print("remuneracao:", df_remuneracao.count() - df_remuneracao.dropna().count(), "linhas com nulo")
print("senioridade:", df_senioridade.count() - df_senioridade.dropna().count(), "linhas com nulo")
print("tecnologias:", df_tecnologias.count() - df_tecnologias.dropna().count(), "linhas com nulo")
print("uso_ia:", df_uso_ia.count() - df_uso_ia.dropna().count(), "linhas com nulo")

cargo: 0 linhas com nulo
genero: 0 linhas com nulo
modelo_trabalho: 0 linhas com nulo
regiao: 0 linhas com nulo
remuneracao: 0 linhas com nulo
senioridade: 0 linhas com nulo
tecnologias: 6 linhas com nulo
uso_ia: 6 linhas com nulo


In [9]:
# Olhando as linhas que têm valor nulo na coluna percentual_respostas_validas

print(' TECNOLOGIAS ')
df_tecnologias.filter(df_tecnologias["percentual_respostas_validas"].isNull()).show(truncate=False)

print(' USO DE IA ')
df_uso_ia.filter(df_uso_ia["percentual_respostas_validas"].isNull()).show(truncate=False)

 TECNOLOGIAS 
+------------+----------+-------------+----------+-------------------+-----------------------+----------------+----------------------------+
|ano_pesquisa|tecnologia|classificacao|quantidade|total_participantes|total_respostas_validas|percentual_total|percentual_respostas_validas|
+------------+----------+-------------+----------+-------------------+-----------------------+----------------+----------------------------+
|2023        |SQL       |Nao respondeu|1521      |5293               |3772                   |28.74           |NULL                        |
|2023        |Python    |Nao respondeu|1521      |5293               |3772                   |28.74           |NULL                        |
|2024        |SQL       |Nao respondeu|1628      |5217               |3589                   |31.21           |NULL                        |
|2024        |Python    |Nao respondeu|1628      |5217               |3589                   |31.21           |NULL                        |

**Sobre os valores nulos:** só `perfil_tecnologias` e `perfil_uso_ia` têm nulos, 6 linhas em cada, sempre na coluna `percentual_respostas_validas`. Isso é esperado: quem aparece como "Nao respondeu" (ou "Resposta conflitante", no caso de IA) fica de fora da base de respostas válidas, então não tem percentual nessa coluna. Não é erro de dado, e por isso vamos manter essas linhas.

In [10]:
# Conferindo se a soma dos percentuais de cada pesquisa fecha em 100%
# F.sum soma os valores e F.round arredonda para 2 casas decimais

print(' CARGO ')
df_cargo.groupBy("ano_pesquisa").agg(F.round(F.sum("percentual"), 2).alias("soma_percentual")).show()

print(' GENERO ')
df_genero.groupBy("ano_pesquisa").agg(F.round(F.sum("percentual"), 2).alias("soma_percentual")).show()

print(' MODELO DE TRABALHO ')
df_modelo_trabalho.groupBy("ano_pesquisa").agg(F.round(F.sum("percentual"), 2).alias("soma_percentual")).show()

print(' REGIAO ')
df_regiao.groupBy("ano_pesquisa").agg(F.round(F.sum("percentual"), 2).alias("soma_percentual")).show()

print(' SENIORIDADE ')
df_senioridade.groupBy("ano_pesquisa").agg(F.round(F.sum("percentual"), 2).alias("soma_percentual")).show()

print(' REMUNERACAO (por senioridade) ')
df_remuneracao.groupBy("ano_pesquisa", "senioridade").agg(F.round(F.sum("percentual"), 2).alias("soma_percentual")).show()

print(' TECNOLOGIAS (percentual_total) ')
df_tecnologias.groupBy("ano_pesquisa", "tecnologia").agg(F.round(F.sum("percentual_total"), 2).alias("soma_percentual")).show()

print(' USO DE IA (percentual_total) ')
df_uso_ia.groupBy("ano_pesquisa").agg(F.round(F.sum("percentual_total"), 2).alias("soma_percentual")).show()

 CARGO 
+------------+---------------+
|ano_pesquisa|soma_percentual|
+------------+---------------+
|   2025_2026|         100.01|
|        2023|          99.99|
|        2024|         100.01|
+------------+---------------+

 GENERO 
+------------+---------------+
|ano_pesquisa|soma_percentual|
+------------+---------------+
|   2025_2026|          99.99|
|        2023|          100.0|
|        2024|          100.0|
+------------+---------------+

 MODELO DE TRABALHO 
+------------+---------------+
|ano_pesquisa|soma_percentual|
+------------+---------------+
|   2025_2026|          100.0|
|        2023|          100.0|
|        2024|          100.0|
+------------+---------------+

 REGIAO 
+------------+---------------+
|ano_pesquisa|soma_percentual|
+------------+---------------+
|   2025_2026|          99.99|
|        2023|         100.01|
|        2024|          100.0|
+------------+---------------+

 SENIORIDADE 
+------------+---------------+
|ano_pesquisa|soma_percentual|
+----

In [11]:
# Conferindo se a soma das quantidades bate com o total de respondentes
# F.sum soma as quantidades e F.max pega o total (ele se repete em todas as linhas do grupo)

print(' REGIAO ')
df_regiao.groupBy("ano_pesquisa").agg(
    F.sum("quantidade").alias("soma_quantidade"),
    F.max("total_respostas_validas").alias("total_informado")
).show()

print(' REMUNERACAO (por senioridade) ')
df_remuneracao.groupBy("ano_pesquisa", "senioridade").agg(
    F.sum("quantidade").alias("soma_quantidade"),
    F.max("total_respostas_validas").alias("total_informado")
).show()

 REGIAO 
+------------+---------------+---------------+
|ano_pesquisa|soma_quantidade|total_informado|
+------------+---------------+---------------+
|   2025_2026|           3370|           3370|
|        2023|           5169|           5169|
|        2024|           5075|           5075|
+------------+---------------+---------------+

 REMUNERACAO (por senioridade) 
+------------+-------------------+---------------+---------------+
|ano_pesquisa|        senioridade|soma_quantidade|total_informado|
+------------+-------------------+---------------+---------------+
|   2025_2026|             Sênior|            858|            858|
|        2024|             Júnior|            868|            868|
|   2025_2026|Especialista/Staff+|            349|            349|
|   2025_2026|              Pleno|            776|            776|
|        2023|             Júnior|           1046|           1046|
|        2023|             Sênior|           1419|           1419|
|        2024|            

**Sobre a qualidade dos dados:** conferimos três pontos nas 8 tabelas da camada Gold.

1. **Valores nulos:** os únicos são 12 linhas (6 em `perfil_tecnologias` e 6 em `perfil_uso_ia`), de quem não respondeu à pergunta. Não é erro de dado.
2. **Quantidades:** em todas as pesquisas, a soma das quantidades é igual ao total de respondentes da pergunta.
3. **Percentuais:** somam 100%, com diferença de no máximo 0,03 ponto percentual (na remuneração). Isso vem do arredondamento a duas casas decimais, não de erro.

A camada Gold está consistente e pode ser usada nas análises.

# Estrutura do Mercado

Esta trilha mostra como o mercado de dados está organizado: quais cargos existem e qual o peso de cada nível de senioridade.

As perguntas que vamos responder:

1. Quais são os cargos mais frequentes no mercado de dados?
2. Como o mix de cargos mudou entre 2023 e 2025/26?
3. Qual o peso de cada nível de senioridade?

In [12]:
# Filtra só a pesquisa mais recente (2025_2026)
cargos_atual = df_cargo.filter(df_cargo["ano_pesquisa"] == "2025_2026")

# Ordena do maior percentual para o menor e mostra os 10 primeiros
cargos_atual.select("cargo", "quantidade", "percentual").orderBy("percentual", ascending=False).show(10, truncate=False)

+-----------------------------------------------------------+----------+----------+
|cargo                                                      |quantidade|percentual|
+-----------------------------------------------------------+----------+----------+
|Analista de Dados/Data Analyst                             |599       |23.95     |
|Engenharia e Arquitetura de Dados                          |430       |17.19     |
|Cientista de Dados/Data Scientist                          |424       |16.95     |
|Analista de BI/BI Analyst                                  |215       |8.6       |
|Outra Opção                                                |206       |8.24      |
|Analista de Negócios/Business Analyst                      |141       |5.64      |
|Analytics Engineer                                         |135       |5.4       |
|Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas|106       |4.24      |
|Engenheiro de Machine Learning/ML Engineer/AI Engineer     |106       |4.24

In [13]:
# Pega os 10 cargos mais frequentes da pesquisa mais recente e converte para o pandas
# (o gráfico do Plotly usa tabela do pandas)
top10_cargos = cargos_atual.orderBy("percentual", ascending=False).limit(10).toPandas()

# Texto que vai aparecer na ponta de cada barra (ex.: 24,0%)
top10_cargos["rotulo"] = top10_cargos["percentual"].round(1).astype(str).str.replace(".", ",", regex=False) + "%"

# Cria a figura vazia
fig_cargos = go.Figure()

# Adiciona as barras horizontais (azul-marinho)
fig_cargos.add_trace(
    go.Bar(
        x=top10_cargos["percentual"],
        y=top10_cargos["cargo"],
        orientation="h",
        marker=dict(color="#000A53"),
        text=top10_cargos["rotulo"],
        textposition="outside"
    )
)

# Configura o layout
fig_cargos.update_layout(
    title=dict(text="Os 10 Cargos Mais Frequentes (pesquisa 2025/26)", x=0.5, xanchor="center"),
    xaxis=dict(title="% dos profissionais", ticksuffix="%", range=[0, 30]),
    yaxis=dict(autorange="reversed", automargin=True),
    template="plotly_white",
    showlegend=False,
    width=1200,
    height=600
)

fig_cargos.show()

**Cargos mais frequentes:**

Três cargos concentram mais de 58% dos profissionais: Analista de Dados (cerca de 24%), Engenharia e Arquitetura de Dados (17%) e Cientista de Dados (17%). O Analista de Dados é o cargo mais comum, com praticamente um em cada quatro profissionais. Em seguida aparecem Analista de BI (8,6%) e a opção "Outra" (8,2%). Os cargos de Machine Learning e de Desenvolvimento de Software aparecem com 4,2% cada.

In [14]:
# Compara a participação de cada cargo nas 3 pesquisas
# groupBy + pivot transforma cada ano em uma coluna (como o pivot_table do pandas)
cargos_por_ano = df_cargo.groupBy("cargo").pivot("ano_pesquisa").agg(F.first("percentual"))

# Remove os 3 cargos que não aparecem em todas as pesquisas (só existem em 2023)
cargos_por_ano = cargos_por_ano.dropna()

# Calcula quanto cada cargo ganhou ou perdeu entre 2023 e 2025_2026 (em pontos percentuais)
cargos_por_ano = cargos_por_ano.withColumn("variacao_pp", F.round(F.col("2025_2026") - F.col("2023"), 2))

# Mostra do que mais ganhou para o que mais perdeu
cargos_por_ano.orderBy("variacao_pp", ascending=False).show(truncate=False)

+-----------------------------------------------------------+-----+-----+---------+-----------+
|cargo                                                      |2023 |2024 |2025_2026|variacao_pp|
+-----------------------------------------------------------+-----+-----+---------+-----------+
|Engenheiro de Machine Learning/ML Engineer/AI Engineer     |1.94 |2.7  |4.24     |2.3        |
|Analytics Engineer                                         |3.6  |5.97 |5.4      |1.8        |
|Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas|2.72 |3.2  |4.24     |1.52       |
|Outra Opção                                                |6.79 |6.65 |8.24     |1.45       |
|Analista de Negócios/Business Analyst                      |5.06 |4.82 |5.64     |0.58       |
|Analista de Dados/Data Analyst                             |23.52|25.07|23.95    |0.43       |
|Analista de Suporte/Analista Técnico                       |1.87 |2.07 |2.2      |0.33       |
|Outras Engenharias (não inclui dev)    

In [15]:
# Converte a tabela de variação para o pandas (o maior ganho fica no topo)
variacao_cargos = cargos_por_ano.orderBy("variacao_pp", ascending=False).toPandas()

# Cor de cada barra: azul-marinho para quem ganhou participação, vinho para quem perdeu
variacao_cargos["cor"] = np.where(variacao_cargos["variacao_pp"] >= 0, "#000A53", "#9b1c31")

# Cria a figura vazia
fig_variacao = go.Figure()

# Adiciona as barras horizontais
fig_variacao.add_trace(
    go.Bar(
        x=variacao_cargos["variacao_pp"],
        y=variacao_cargos["cargo"],
        orientation="h",
        marker=dict(color=variacao_cargos["cor"]),
        texttemplate="%{x:+.1f}",
        textposition="outside"
    )
)

# Configura o layout
fig_variacao.update_layout(
    title=dict(text="Variação da Participação de Cada Cargo (2023 para 2025/26)", x=0.5, xanchor="center"),
    xaxis=dict(title="Variação em pontos percentuais (pp)", range=[-6, 3.5]),
    yaxis=dict(autorange="reversed", automargin=True),
    template="plotly_white",
    showlegend=False,
    width=1200,
    height=700
)

fig_variacao.show()

**Mudança no mix de cargos:**

Entre 2023 e 2025/26, o cargo que mais perdeu espaço foi o de Analista de BI, que caiu de 13,1% para 8,6% (-4,5 pp). Em sentido contrário, o Engenheiro de Machine Learning mais que dobrou de peso, de 1,9% para 4,2% (+2,3 pp), e Analytics Engineer (+1,8 pp) e Desenvolvedor de Software (+1,5 pp) também ganharam participação. Os cargos mais comuns, Analista de Dados e Engenharia de Dados, ficaram praticamente estáveis (+0,4 pp e -0,5 pp). O mercado parece se deslocar de funções voltadas a relatórios para funções de engenharia e de machine learning.

In [16]:
# Participação de cada nível de senioridade nas 3 pesquisas
senioridade_por_ano = df_senioridade.groupBy("senioridade").pivot("ano_pesquisa").agg(F.first("percentual"))

# Mostra a tabela (NULL = nível que não existia naquela pesquisa)
senioridade_por_ano.show(truncate=False)

+-------------------+-----+-----+---------+
|senioridade        |2023 |2024 |2025_2026|
+-------------------+-----+-----+---------+
|Especialista/Staff+|NULL |NULL |13.95    |
|Júnior             |27.12|22.73|20.71    |
|Sênior             |36.79|41.2 |34.31    |
|Pleno              |36.09|36.07|31.03    |
+-------------------+-----+-----+---------+



In [17]:
# Converte para o pandas e coloca os níveis na ordem da carreira
senioridade_pd = senioridade_por_ano.toPandas().set_index("senioridade")
senioridade_pd = senioridade_pd.loc[["Júnior", "Pleno", "Sênior", "Especialista/Staff+"]]

# Cria a figura vazia
fig_senioridade = go.Figure()

# Uma barra para cada pesquisa (do mais claro, a mais antiga, ao mais escuro, a mais recente)
fig_senioridade.add_trace(
    go.Bar(x=senioridade_pd.index, y=senioridade_pd["2023"], name="2023",
           marker=dict(color="#7ba7d9"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_senioridade.add_trace(
    go.Bar(x=senioridade_pd.index, y=senioridade_pd["2024"], name="2024",
           marker=dict(color="#3d5a96"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_senioridade.add_trace(
    go.Bar(x=senioridade_pd.index, y=senioridade_pd["2025_2026"], name="2025/26",
           marker=dict(color="#000A53"), texttemplate="%{y:.1f}%", textposition="outside")
)

# Configura o layout
fig_senioridade.update_layout(
    title=dict(text="Distribuição por Senioridade (2023, 2024 e 2025/26)", x=0.5, xanchor="center"),
    xaxis=dict(title="Nível de senioridade"),
    yaxis=dict(title="% dos profissionais", ticksuffix="%", range=[0, 50]),
    barmode="group",
    template="plotly_white",
    width=1200,
    height=600
)

fig_senioridade.show()

**Distribuição por senioridade:**

Em todas as pesquisas, os juniores são o menor grupo entre os três níveis tradicionais, e a participação deles caiu de 27,1% (2023) para 20,7% (2025/26). Atenção: a categoria Especialista/Staff+ só existe na pesquisa mais recente (cerca de 14%), então a queda de Pleno e Sênior em 2025/26 não pode ser lida só como mudança do mercado, porque parte dela vem dessa nova categoria. Somando Sênior e Especialista/Staff+, a participação sobe de 36,8% (2023) para 48,3% (2025/26).

## Conclusão da Trilha 1: Estrutura do Mercado

1. **O mercado é concentrado em poucos cargos.** Analista de Dados, Engenharia e Arquitetura de Dados e Cientista de Dados somam mais de 58% dos profissionais.

2. **O mix está se deslocando para engenharia e machine learning.** O Analista de BI perdeu 4,5 pp desde 2023, enquanto Engenheiro de Machine Learning (+2,3 pp), Analytics Engineer (+1,8 pp) e Desenvolvedor de Software (+1,5 pp) ganharam espaço.

3. **A base da carreira está mais estreita.** A participação de juniores caiu de 27,1% para 20,7%, e Sênior somado a Especialista/Staff+ chegou a 48,3% em 2025/26 (com o cuidado da nova categoria).

**Implicação estratégica:** ao montar um time de dados, a instituição encontra muitos analistas no mercado, mas as funções de engenharia e de machine learning ganham peso e os perfis sêniores são cada vez mais frequentes. Formar juniores internamente pode ser uma alternativa para ampliar o time sem depender só da disputa por profissionais experientes.

# Remuneração e Valorização

Esta trilha mostra quanto o mercado paga em cada nível de senioridade. A camada Gold não cruza cargo com remuneração, então a análise é feita por senioridade, e os valores são nominais (sem correção pela inflação).

As perguntas que vamos responder:

1. Qual a faixa salarial mais comum em cada nível de senioridade?
2. Quanto a remuneração cresce de um nível para o outro?
3. Quantos profissionais de cada nível ganham a partir de R$ 12 mil?

In [18]:
# Filtra só a pesquisa mais recente (2025_2026)
rem_atual = df_remuneracao.filter(df_remuneracao["ano_pesquisa"] == "2025_2026")

# Para cada nível de senioridade, mostra as 3 faixas salariais mais comuns
for nivel in ["Júnior", "Pleno", "Sênior", "Especialista/Staff+"]:
    print(f" {nivel} ")
    rem_atual.filter(rem_atual["senioridade"] == nivel).orderBy("percentual", ascending=False).select("faixa_salarial", "percentual").show(3, truncate=False)

 Júnior 
+------------------------------+----------+
|faixa_salarial                |percentual|
+------------------------------+----------+
|de R$ 4.001/mês a R$ 6.000/mês|23.55     |
|de R$ 1.001/mês a R$ 2.000/mês|19.31     |
|de R$ 2.001/mês a R$ 3.000/mês|18.92     |
+------------------------------+----------+
only showing top 3 rows
 Pleno 
+-------------------------------+----------+
|faixa_salarial                 |percentual|
+-------------------------------+----------+
|de R$ 8.001/mês a R$ 12.000/mês|32.73     |
|de R$ 6.001/mês a R$ 8.000/mês |25.0      |
|de R$ 4.001/mês a R$ 6.000/mês |20.88     |
+-------------------------------+----------+
only showing top 3 rows
 Sênior 
+--------------------------------+----------+
|faixa_salarial                  |percentual|
+--------------------------------+----------+
|de R$ 8.001/mês a R$ 12.000/mês |32.4      |
|de R$ 12.001/mês a R$ 16.000/mês|23.54     |
|de R$ 16.001/mês a R$ 20.000/mês|13.64     |
+--------------------------

**Faixa salarial mais comum de cada nível:**

Em 2025/26, a faixa mais comum dos juniores é de R$ 4.001 a R$ 6.000 (24%). Plenos e seniores têm a mesma faixa mais comum, de R$ 8.001 a R$ 12.000 (33% e 32%), mas os seniores têm uma parcela muito maior de salários acima disso, como veremos a seguir. Entre os Especialistas/Staff+, a faixa mais comum sobe para R$ 12.001 a R$ 16.000 (26%).

In [19]:
# Lista das faixas salariais a partir de R$ 12.001 por mês
faixas_altas = [
    "de R$ 12.001/mês a R$ 16.000/mês",
    "de R$ 16.001/mês a R$ 20.000/mês",
    "de R$ 20.001/mês a R$ 25.000/mês",
    "de R$ 25.001/mês a R$ 30.000/mês",
    "de R$ 30.001/mês a R$ 40.000/mês",
    "Acima de R$ 40.001/mês"
]

# Fica só com essas faixas, soma os percentuais de cada nível
# e coloca cada pesquisa em uma coluna (pivot)
salario_alto = (
    df_remuneracao
    .filter(df_remuneracao["faixa_salarial"].isin(faixas_altas))
    .groupBy("senioridade")
    .pivot("ano_pesquisa")
    .agg(F.round(F.sum("percentual"), 1))
)

salario_alto.show(truncate=False)

+-------------------+----+----+---------+
|senioridade        |2023|2024|2025_2026|
+-------------------+----+----+---------+
|Especialista/Staff+|NULL|NULL|83.7     |
|Júnior             |0.8 |1.4 |1.5      |
|Sênior             |49.6|56.8|52.1     |
|Pleno              |8.5 |9.3 |11.1     |
+-------------------+----+----+---------+



In [20]:
# Converte para o pandas e coloca os níveis na ordem da carreira
salario_alto_pd = salario_alto.toPandas().set_index("senioridade")
salario_alto_pd = salario_alto_pd.loc[["Júnior", "Pleno", "Sênior", "Especialista/Staff+"]]

# Cria a figura vazia
fig_salario = go.Figure()

# Uma barra para cada pesquisa (do mais claro, a mais antiga, ao mais escuro, a mais recente)
fig_salario.add_trace(
    go.Bar(x=salario_alto_pd.index, y=salario_alto_pd["2023"], name="2023",
           marker=dict(color="#7ba7d9"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_salario.add_trace(
    go.Bar(x=salario_alto_pd.index, y=salario_alto_pd["2024"], name="2024",
           marker=dict(color="#3d5a96"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_salario.add_trace(
    go.Bar(x=salario_alto_pd.index, y=salario_alto_pd["2025_2026"], name="2025/26",
           marker=dict(color="#000A53"), texttemplate="%{y:.1f}%", textposition="outside")
)

# Configura o layout
fig_salario.update_layout(
    title=dict(text="Profissionais que Ganham a Partir de R$ 12.001 por Mês", x=0.5, xanchor="center"),
    xaxis=dict(title="Nível de senioridade"),
    yaxis=dict(title="% dos profissionais do nível", ticksuffix="%", range=[0, 100]),
    barmode="group",
    template="plotly_white",
    width=1200,
    height=600
)

fig_salario.show()

**Quem ganha a partir de R$ 12 mil:**

O salto entre os níveis é grande. Em 2025/26, apenas 1,5% dos juniores e 11,1% dos plenos ganham a partir de R$ 12.001 por mês, contra 52,1% dos seniores e 83,7% dos Especialistas/Staff+. Entre os seniores, mais da metade já ultrapassa os R$ 12 mil, embora a faixa mais comum seja a de R$ 8.001 a R$ 12.000. A parcela de seniores nessa faixa subiu de 49,6% (2023) para 56,8% (2024) e recuou para 52,1% em 2025/26. Atenção: os valores são nominais, sem correção pela inflação.

## Conclusão da Trilha 2: Remuneração e Valorização

1. **A remuneração cresce muito com a senioridade.** Em 2025/26, ganham a partir de R$ 12.001 por mês apenas 1,5% dos juniores e 11,1% dos plenos, contra 52,1% dos seniores e 83,7% dos Especialistas/Staff+.

2. **A faixa mais comum sobe com o nível.** Júnior: R$ 4.001 a R$ 6.000. Pleno e Sênior: R$ 8.001 a R$ 12.000. Especialista/Staff+: R$ 12.001 a R$ 16.000. O Sênior se diferencia pela parcela grande de salários acima de R$ 12 mil.

3. **A parcela de seniores com salário alto oscilou.** Foi de 49,6% (2023) para 56,8% (2024) e 52,1% (2025/26), então não houve crescimento contínuo.

**Implicação estratégica:** a progressão de carreira é a principal alavanca de remuneração. Para a instituição, o salto de pleno para sênior é o ponto mais caro do mercado, e desenvolver plenos internamente pode ser uma alternativa a disputar seniores já formados. Limites: a Gold não cruza cargo com salário, e os valores são nominais (sem correção pela inflação).

# Perfil dos Profissionais

Esta trilha mostra quem são os profissionais de dados: o gênero, a região onde vivem e como trabalham.

As perguntas que vamos responder:

1. Qual a participação feminina no mercado de dados e como ela evoluiu?
2. Os profissionais estão distribuídos pelo país na mesma proporção da população?
3. Como mudou o modelo de trabalho (remoto, híbrido e presencial)?

In [21]:
# Mostra a tabela de gênero completa (3 pesquisas)
df_genero.orderBy("ano_pesquisa", "genero").show(truncate=False)

+------------+--------------------+----------+-----------------------+----------+
|ano_pesquisa|genero              |quantidade|total_respostas_validas|percentual|
+------------+--------------------+----------+-----------------------+----------+
|2023        |Feminino            |1293      |5293                   |24.43     |
|2023        |Masculino           |3975      |5293                   |75.1      |
|2023        |Outro               |9         |5293                   |0.17      |
|2023        |Prefiro não informar|16        |5293                   |0.3       |
|2024        |Feminino            |1226      |5217                   |23.5      |
|2024        |Masculino           |3968      |5217                   |76.06     |
|2024        |Outro               |8         |5217                   |0.15      |
|2024        |Prefiro não informar|15        |5217                   |0.29      |
|2025_2026   |Feminino            |768       |3495                   |21.97     |
|2025_2026   |Ma

In [22]:
# Fica só com o gênero feminino (uma linha por pesquisa) e converte para o pandas
feminino = df_genero.filter(df_genero["genero"] == "Feminino").orderBy("ano_pesquisa").toPandas()

# Cria a figura vazia
fig_genero = go.Figure()

# Linha com a participação feminina (vinho)
fig_genero.add_trace(
    go.Scatter(
        x=feminino["ano_pesquisa"],
        y=feminino["percentual"],
        mode="lines+markers+text",
        line=dict(color="#9b1c31", width=3),
        marker=dict(size=10),
        texttemplate="%{y:.1f}%",
        textposition="top center"
    )
)

# Configura o layout
fig_genero.update_layout(
    title=dict(text="Participação Feminina entre os Participantes (2023 a 2025/26)", x=0.5, xanchor="center"),
    xaxis=dict(title="Pesquisa", type="category"),
    yaxis=dict(title="% de mulheres", ticksuffix="%", range=[0, 35]),
    template="plotly_white",
    showlegend=False,
    width=1200,
    height=600
)

fig_genero.show()

**Participação feminina:**

A participação feminina caiu de 24,4% (2023) para 22,0% (2025/26), uma queda de 2,5 pp. Em 2025/26, mais de 77% dos participantes são homens. Como a pesquisa é de adesão voluntária, vale ler o resultado como uma tendência e não como um número exato do mercado inteiro.

In [23]:
# Participação de cada região na pesquisa mais recente (2025_2026), convertida para o pandas
regiao_atual = df_regiao.filter(df_regiao["ano_pesquisa"] == "2025_2026").select("regiao", "percentual").toPandas()

# Participação de cada região na população brasileira (IBGE, Censo 2022, valores aproximados)
populacao = pd.DataFrame({
    "regiao": ["Sudeste", "Nordeste", "Sul", "Norte", "Centro-oeste"],
    "pct_populacao": [41.8, 26.9, 14.7, 8.5, 8.0]
})

# Junta as duas tabelas pela região e calcula o índice de representação
comparacao = regiao_atual.merge(populacao, on="regiao")
comparacao["indice"] = (comparacao["percentual"] / comparacao["pct_populacao"]).round(2)
comparacao = comparacao.sort_values("percentual", ascending=False)
comparacao

,regiao,percentual,pct_populacao,indice
0,Sudeste,64.42,41.8,1.54
3,Sul,16.02,14.7,1.09
1,Nordeste,11.28,26.9,0.42
4,Centro-oeste,6.85,8.0,0.86
2,Norte,1.42,8.5,0.17


In [24]:
# Cria a figura vazia
fig_regiao = go.Figure()

# Barras dos profissionais de dados (azul-marinho)
fig_regiao.add_trace(
    go.Bar(x=comparacao["regiao"], y=comparacao["percentual"], name="Profissionais de dados (2025/26)",
           marker=dict(color="#000A53"), texttemplate="%{y:.1f}%", textposition="outside")
)

# Barras da população brasileira (azul claro)
fig_regiao.add_trace(
    go.Bar(x=comparacao["regiao"], y=comparacao["pct_populacao"], name="População brasileira (Censo 2022)",
           marker=dict(color="#7ba7d9"), texttemplate="%{y:.1f}%", textposition="outside")
)

# Configura o layout
fig_regiao.update_layout(
    title=dict(text="Onde Estão os Profissionais de Dados versus Onde Vive a População", x=0.5, xanchor="center"),
    xaxis=dict(title="Região"),
    yaxis=dict(title="% do total", ticksuffix="%", range=[0, 75]),
    barmode="group",
    template="plotly_white",
    width=1200,
    height=600
)

fig_regiao.show()

**Onde estão os profissionais:**

O Sudeste concentra 64,4% dos profissionais de dados, bem acima do seu peso na população (41,8%). O Nordeste faz o caminho inverso: tem 26,9% da população, mas só 11,3% dos profissionais. O Norte também fica muito abaixo (1,4% dos profissionais contra 8,5% da população). O Sul está proporcional (16,0% contra 14,7%) e o Centro-Oeste um pouco abaixo (cerca de 7% contra 8%). O Sudeste ainda ganhou espaço: foi de 61,4% em 2023 para 64,4% em 2025/26. Os dados de população vêm do Censo 2022 do IBGE.

In [25]:
# Participação de cada modelo de trabalho nas 3 pesquisas (cada pesquisa vira uma coluna)
modelo_por_ano = df_modelo_trabalho.groupBy("modelo_trabalho").pivot("ano_pesquisa").agg(F.first("percentual"))

modelo_por_ano.show(truncate=False)

+---------------+-----+-----+---------+
|modelo_trabalho|2023 |2024 |2025_2026|
+---------------+-----+-----+---------+
|Presencial     |16.62|16.29|20.76    |
|Remoto         |46.31|45.69|39.71    |
|Hibrido        |37.07|38.02|39.53    |
+---------------+-----+-----+---------+



In [26]:
# Converte para o pandas e coloca na ordem Remoto, Híbrido, Presencial
modelo_pd = modelo_por_ano.toPandas().set_index("modelo_trabalho")
modelo_pd = modelo_pd.loc[["Remoto", "Hibrido", "Presencial"]]
modelo_pd.index = ["Remoto", "Híbrido", "Presencial"]   # nomes com acento para o gráfico

# Cria a figura vazia
fig_modelo = go.Figure()

# Uma barra para cada pesquisa (do mais claro ao mais escuro)
fig_modelo.add_trace(
    go.Bar(x=modelo_pd.index, y=modelo_pd["2023"], name="2023",
           marker=dict(color="#7ba7d9"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_modelo.add_trace(
    go.Bar(x=modelo_pd.index, y=modelo_pd["2024"], name="2024",
           marker=dict(color="#3d5a96"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_modelo.add_trace(
    go.Bar(x=modelo_pd.index, y=modelo_pd["2025_2026"], name="2025/26",
           marker=dict(color="#000A53"), texttemplate="%{y:.1f}%", textposition="outside")
)

# Configura o layout
fig_modelo.update_layout(
    title=dict(text="Modelo de Trabalho dos Profissionais (2023, 2024 e 2025/26)", x=0.5, xanchor="center"),
    xaxis=dict(title="Modelo de trabalho"),
    yaxis=dict(title="% dos profissionais", ticksuffix="%", range=[0, 55]),
    barmode="group",
    template="plotly_white",
    width=1200,
    height=600
)

fig_modelo.show()

**Modelo de trabalho:**

O trabalho remoto perdeu espaço: caiu de 46,3% (2023) para 39,7% (2025/26), uma queda de 6,6 pp. No mesmo período o presencial subiu de 16,6% para 20,8% (+4,1 pp) e o híbrido de 37,1% para 39,5% (+2,5 pp). Em 2025/26, remoto e híbrido estão praticamente empatados, e somados reúnem 79,2% dos profissionais, ante 83,4% em 2023.

## Conclusão da Trilha 3: Perfil dos Profissionais

1. **A diversidade de gênero é baixa e vem caindo.** As mulheres eram 24,4% dos participantes em 2023 e são 22,0% em 2025/26 (-2,5 pp). Mais de 77% dos participantes são homens.

2. **O mercado é concentrado no Sudeste.** A região tem 64,4% dos profissionais e 41,8% da população. O Nordeste tem 11,3% dos profissionais para 26,9% da população, e o Norte 1,4% para 8,5%.

3. **O trabalho remoto perdeu espaço.** Caiu de 46,3% para 39,7%, enquanto o presencial subiu de 16,6% para 20,8%. Mesmo assim, remoto e híbrido somam 79,2% dos profissionais.

**Implicação estratégica:** a instituição encontra poucas mulheres e poucos profissionais fora do Sudeste, então metas de diversidade exigem busca ativa. Oferecer trabalho híbrido ou remoto pode ampliar o alcance das contratações, principalmente no Nordeste e no Norte. Limite: a Gold não cruza gênero, região ou modelo de trabalho com cargo ou salário, então não dá para saber se há diferença de remuneração entre esses grupos.

# Tecnologias e Inteligência Artificial

Esta trilha mostra quais tecnologias os profissionais usam e como a Inteligência Artificial entrou no dia a dia. A camada Gold traz apenas SQL e Python, então a análise de tecnologias se limita a essas duas.

As perguntas que vamos responder:

1. Como evoluiu o uso de Python e SQL entre os profissionais?
2. Qual o nível de adoção de Inteligência Artificial?
3. O que esses números realmente conseguem afirmar?

In [27]:
# Fica só com quem utiliza cada tecnologia
tec_utiliza = df_tecnologias.filter(df_tecnologias["classificacao"] == "Utiliza")

# Mostra o percentual entre quem respondeu e entre todos os participantes
tec_utiliza.select("ano_pesquisa", "tecnologia", "quantidade", "percentual_respostas_validas", "percentual_total").orderBy("tecnologia", "ano_pesquisa").show(truncate=False)

+------------+----------+----------+----------------------------+----------------+
|ano_pesquisa|tecnologia|quantidade|percentual_respostas_validas|percentual_total|
+------------+----------+----------+----------------------------+----------------+
|2023        |Python    |2825      |74.89                       |53.37           |
|2024        |Python    |2935      |81.78                       |56.26           |
|2025_2026   |Python    |1929      |92.03                       |55.19           |
|2023        |SQL       |3156      |83.67                       |59.63           |
|2024        |SQL       |3146      |87.66                       |60.3            |
|2025_2026   |SQL       |1764      |84.16                       |50.47           |
+------------+----------+----------+----------------------------+----------------+



In [28]:
# Uso de Python e SQL entre quem respondeu (cada pesquisa vira uma coluna)
tec_por_ano = tec_utiliza.groupBy("tecnologia").pivot("ano_pesquisa").agg(F.first("percentual_respostas_validas"))

# Converte para o pandas e coloca na ordem Python, SQL
tec_pd = tec_por_ano.toPandas().set_index("tecnologia")
tec_pd = tec_pd.loc[["Python", "SQL"]]

# Cria a figura vazia
fig_tec = go.Figure()

# Uma barra para cada pesquisa (do mais claro ao mais escuro)
fig_tec.add_trace(
    go.Bar(x=tec_pd.index, y=tec_pd["2023"], name="2023",
           marker=dict(color="#7ba7d9"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_tec.add_trace(
    go.Bar(x=tec_pd.index, y=tec_pd["2024"], name="2024",
           marker=dict(color="#3d5a96"), texttemplate="%{y:.1f}%", textposition="outside")
)
fig_tec.add_trace(
    go.Bar(x=tec_pd.index, y=tec_pd["2025_2026"], name="2025/26",
           marker=dict(color="#000A53"), texttemplate="%{y:.1f}%", textposition="outside")
)

# Configura o layout
fig_tec.update_layout(
    title=dict(text="Uso de Python e SQL entre Quem Respondeu (2023, 2024 e 2025/26)", x=0.5, xanchor="center"),
    xaxis=dict(title="Tecnologia"),
    yaxis=dict(title="% de quem respondeu", ticksuffix="%", range=[0, 105]),
    barmode="group",
    template="plotly_white",
    width=1200,
    height=600
)

fig_tec.show()

**Adoção de Python e SQL:**

Entre quem respondeu, o uso de Python cresceu em todas as pesquisas, de 74,9% (2023) para 92,0% (2025/26). O SQL ficou estável em um patamar alto, entre 83,7% e 87,7% (84,2% em 2025/26). Atenção à base de cálculo: se considerarmos todos os participantes, incluindo quem não respondeu, o SQL aparece em 59,6% (2023) e 50,5% (2025/26), porque a parcela de pessoas sem resposta à pergunta subiu de 28,7% para 40,0%. Por isso o mais seguro é ler a tendência: Python em alta e SQL estável.

In [29]:
# Mostra a tabela de uso de IA completa (3 pesquisas)
df_uso_ia.select("ano_pesquisa", "classificacao_ia", "quantidade", "percentual_total", "percentual_respostas_validas").orderBy("ano_pesquisa", "classificacao_ia").show(truncate=False)

+------------+--------------------+----------+----------------+----------------------------+
|ano_pesquisa|classificacao_ia    |quantidade|percentual_total|percentual_respostas_validas|
+------------+--------------------+----------+----------------+----------------------------+
|2023        |Nao respondeu       |1521      |28.74           |NULL                        |
|2023        |Nao utiliza         |718       |13.57           |19.17                       |
|2023        |Resposta conflitante|26        |0.49            |NULL                        |
|2023        |Utiliza IA          |3028      |57.21           |80.83                       |
|2024        |Nao respondeu       |1598      |30.63           |NULL                        |
|2024        |Nao utiliza         |224       |4.29            |6.21                        |
|2024        |Resposta conflitante|13        |0.25            |NULL                        |
|2024        |Utiliza IA          |3382      |64.83           |93.79  

In [30]:
# Fica só com quem utiliza IA (uma linha por pesquisa) e converte para o pandas
ia_utiliza = df_uso_ia.filter(df_uso_ia["classificacao_ia"] == "Utiliza IA").orderBy("ano_pesquisa").toPandas()

# Cria a figura vazia
fig_ia = go.Figure()

# Barras entre quem respondeu (azul-marinho)
fig_ia.add_trace(
    go.Bar(x=ia_utiliza["ano_pesquisa"], y=ia_utiliza["percentual_respostas_validas"],
           name="Entre quem respondeu", marker=dict(color="#000A53"),
           texttemplate="%{y:.1f}%", textposition="outside")
)

# Barras entre todos os participantes (azul claro)
fig_ia.add_trace(
    go.Bar(x=ia_utiliza["ano_pesquisa"], y=ia_utiliza["percentual_total"],
           name="Entre todos os participantes", marker=dict(color="#7ba7d9"),
           texttemplate="%{y:.1f}%", textposition="outside")
)

# Configura o layout
fig_ia.update_layout(
    title=dict(text="Percentual que Utiliza IA, Conforme a Base de Cálculo", x=0.5, xanchor="center"),
    xaxis=dict(title="Pesquisa", type="category"),
    yaxis=dict(title="% que utiliza IA", ticksuffix="%", range=[0, 110]),
    barmode="group",
    template="plotly_white",
    width=1200,
    height=600
)

fig_ia.show()

In [31]:
# Soma de quem não respondeu com quem deu resposta conflitante, em cada pesquisa
sem_resposta = df_uso_ia.filter(df_uso_ia["classificacao_ia"].isin(["Nao respondeu", "Resposta conflitante"]))
sem_resposta.groupBy("ano_pesquisa").agg(F.round(F.sum("percentual_total"), 1).alias("sem_resposta_valida")).orderBy("ano_pesquisa").show()

+------------+-------------------+
|ano_pesquisa|sem_resposta_valida|
+------------+-------------------+
|        2023|               29.2|
|        2024|               30.9|
|   2025_2026|               39.9|
+------------+-------------------+



**Adoção de IA:**

Entre quem respondeu, o uso de IA subiu de 80,8% (2023) para 93,8% (2024) e 98,1% (2025/26): quase todos os respondentes usam. Esse número pede cuidado: a parcela de participantes sem resposta válida passou de 29,2% para 39,9%, e, olhando todos os participantes, o uso fica em 57,2%, 64,8% e 59,0%. Além disso, a pesquisa mede a adoção, mas não o impacto da IA (produtividade, remuneração ou cargo), então não dá para afirmar o ganho dela a partir desses dados.

## Conclusão da Trilha 4: Tecnologias e Inteligência Artificial

1. **Python está em alta e SQL se mantém alto.** Entre quem respondeu, o Python foi de 74,9% para 92,0% e o SQL ficou entre 83,7% e 87,7%.

2. **A adoção de IA é muito alta entre os respondentes.** O uso foi de 80,8% para 98,1%, mas 39,9% dos participantes ficaram sem resposta válida em 2025/26.

3. **O aumento das respostas em branco limita a leitura.** A parcela sem resposta subiu de cerca de 29% para cerca de 40% tanto em tecnologias quanto em IA, então as tendências são mais confiáveis que os valores exatos.

**Implicação estratégica:** Python e SQL formam a base mínima de qualquer plano de capacitação em dados, e a IA já faz parte do dia a dia dos profissionais que responderam. Como a pesquisa não mede o impacto da IA, a instituição precisaria medir esse impacto internamente.

# Oportunidades e Recomendações

## Síntese das descobertas

- **Estrutura do mercado:** Analista de Dados, Engenharia de Dados e Cientista de Dados somam mais de 58% dos profissionais. O mix se desloca para engenharia e machine learning (Analista de BI -4,5 pp, Engenheiro de Machine Learning +2,3 pp).
- **Remuneração:** ganham a partir de R$ 12.001 por mês 1,5% dos juniores, 11,1% dos plenos, 52,1% dos seniores e 83,7% dos Especialistas/Staff+.
- **Perfil dos profissionais:** mulheres são 22,0% dos participantes (eram 24,4% em 2023), o Sudeste concentra 64,4% dos profissionais e o trabalho remoto caiu de 46,3% para 39,7%.
- **Tecnologias e IA:** Python vai de 74,9% para 92,0%, SQL fica estável (84,2%) e 98,1% de quem respondeu usa IA.

## Principais riscos e desafios identificados

1. **Disputa por perfis seniores.** Sênior e Especialista/Staff+ somam 48,3% dos respondentes de 2025/26, e 52,1% dos seniores (83,7% dos Especialistas/Staff+) ganham a partir de R$ 12.001. É a faixa mais cara do mercado.

2. **Diversidade baixa e em queda.** A participação feminina caiu de 24,4% para 22,0% (-2,5 pp). Sem ação ativa, o time tende a refletir esse mercado.

3. **Talento concentrado no Sudeste.** A região tem 64,4% dos profissionais e 41,8% da população. Nordeste (11,3%) e Norte (1,4%) estão muito sub-representados.

4. **Retorno ao presencial.** O trabalho remoto caiu de 46,3% para 39,7% e o presencial subiu de 16,6% para 20,8%. Políticas rígidas de presença podem reduzir o alcance das vagas.

5. **IA sem medição de impacto.** 98,1% dos respondentes usam IA, mas 39,9% dos participantes ficaram sem resposta válida e a pesquisa não mede ganho de produtividade ou de remuneração.

## Recomendações estratégicas

**Contratar**

1. **Montar o time por funções, não só por cargos tradicionais.** Reforçar engenharia de dados e machine learning, que ganharam espaço (Engenheiro de Machine Learning +2,3 pp, Analytics Engineer +1,8 pp), sem abrir mão de uma base de analistas, o cargo mais comum do mercado (cerca de 24%).

2. **Ampliar o alcance geográfico e a diversidade com busca ativa.** Oferecer trabalho híbrido ou remoto e buscar profissionais no Nordeste e no Norte, além de metas para a participação feminina (22,0% no mercado).

**Capacitar**

3. **Desenvolver plenos e juniores internamente.** O salto de remuneração acontece na senioridade (1,5% dos juniores e 11,1% dos plenos ganham a partir de R$ 12.001, contra 52,1% dos seniores). Uma trilha de carreira clara reduz a dependência de seniores já formados.

4. **Python e SQL como base, IA como competência transversal.** Python (92,0%) e SQL (84,2%) são usados pela grande maioria dos respondentes, e 98,1% usam IA.

**Investir em tecnologia**

5. **Priorizar uma plataforma de dados que sustente engenharia e machine learning.** Engenharia de Dados é o segundo cargo mais frequente (17,2%) e Machine Learning mais que dobrou de peso, então são as funções que mais vão depender dessa base.

6. **Criar governança e medir o impacto da IA internamente.** Como a pesquisa mede adoção e não impacto, a instituição precisa de indicadores próprios, e em uma instituição financeira o uso de IA também exige regras de governança.

*Observação: estas recomendações são hipóteses baseadas no mercado, não uma análise de custo e benefício para a instituição.*

## Limitações e próximos passos

- **Cruzamentos que a Gold não tem:** cargo com remuneração, gênero com senioridade ou salário, região ou modelo de trabalho com salário, e uso de IA com cargo ou salário. Com eles, seria possível dizer quais perfis são mais valorizados e se há diferença de remuneração entre grupos.
- **Tecnologias:** a Gold traz só SQL e Python. Uma lista maior de ferramentas permitiria apontar as de maior adoção.
- **Pesquisa de adesão voluntária:** os resultados descrevem quem respondeu, não necessariamente todo o mercado.
- **Mudanças no questionário:** três cargos só existem em 2023 e o nível Especialista/Staff+ só existe em 2025/26.
- **Valores nominais:** as faixas salariais não têm correção pela inflação.
- **Dado externo:** a população por região vem do Censo 2022 do IBGE.

## Conclusão final

O mercado brasileiro de dados está concentrado em poucos cargos e se desloca para engenharia de dados e machine learning. A remuneração depende muito da senioridade, com um salto grande entre pleno e sênior, e a base de tecnologias é clara: Python em alta, SQL estável e IA já presente no dia a dia de quase todos os respondentes.

Para uma instituição financeira que quer investir em Dados, Analytics e IA, o caminho é contratar por funções, capacitar internamente para reduzir a dependência de seniores, ampliar o alcance geográfico e de diversidade e investir em uma plataforma de dados com governança. Os dados mostram a direção do mercado, mas os cruzamentos que faltam na camada Gold são o próximo passo para quantificar o custo e o retorno dessas decisões.

In [32]:
# Salva em arquivos CSV as tabelas que alimentaram os gráficos
top10_cargos.to_csv("saida_top10_cargos.csv", index=False)
variacao_cargos.to_csv("saida_variacao_cargos.csv", index=False)
senioridade_pd.to_csv("saida_senioridade.csv")
salario_alto_pd.to_csv("saida_salario_alto.csv")
feminino.to_csv("saida_participacao_feminina.csv", index=False)
comparacao.to_csv("saida_regiao_vs_populacao.csv", index=False)
modelo_pd.to_csv("saida_modelo_trabalho.csv")
tec_pd.to_csv("saida_tecnologias.csv")
ia_utiliza.to_csv("saida_uso_ia.csv", index=False)

print("Arquivos salvos.")

Arquivos salvos.
